In [1]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd



In [3]:
# Define model names for SciBERT and SapBERT
models = {
    "SciBERT": "allenai/scibert_scivocab_uncased",
    "SapBERT": "cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
}

# Input and output file paths
input_file = "biobert_embedding_terms.csv"  # Replace with your input file path
output_files = {
    "SciBERT": "tumor_embeddings_scibert.csv",
    "SapBERT": "tumor_embeddings_sapbert.csv",
}



In [4]:
# Load tumor names from the CSV file
df = pd.read_csv(input_file)
if "Tumor_Names" not in df.columns:
    raise ValueError("The column 'Tumor_Names' is not found in the input file.")

tumor_names = df["Tumor_Names"].tolist()



In [5]:
# Function to generate embeddings
def generate_embeddings(model_name, tumor_names):
    print(f"Generating embeddings using {model_name}...")
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()  # Set model to evaluation mode

    embeddings = []
    with torch.no_grad():
        for name in tumor_names:
            # Tokenize and process the input
            inputs = tokenizer(name, return_tensors="pt", padding=True, truncation=True)
            outputs = model(**inputs)
            # Use the CLS token embedding
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
            embeddings.append(cls_embedding)

    return embeddings



In [6]:
# Generate and save embeddings for each model
for model_name, output_file in output_files.items():
    embeddings = generate_embeddings(models[model_name], tumor_names)
    
    # Create a DataFrame for embeddings
    embedding_columns = [f"dim_{i}" for i in range(len(embeddings[0]))]
    embeddings_df = pd.DataFrame(embeddings, columns=embedding_columns)
    embeddings_df["Tumor_Names"] = tumor_names

    # Save to CSV
    embeddings_df.to_csv(output_file, index=False)
    print(f"Embeddings saved to {output_file}")

Generating embeddings using allenai/scibert_scivocab_uncased...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

2025-01-23 15:07:37.922901: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-23 15:07:37.962270: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-23 15:07:37.962308: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-23 15:07:37.963621: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-23 15:07:37.970369: I tensorflow/core/platform/cpu_feature_guar

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Embeddings saved to tumor_embeddings_sapbert.csv
